In [ ]:
import os 
import polars as pl
import simple_icd_10_cm as cm
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
from functools import partial
from time import time
from tqdm import tqdm

from note_workers import note_pool_context, process_note_file

DATA_PATH = Path('/data/gusev/USERS/jpconnor/data/')
OncDRS_PATH = Path('/data/gusev/PROFILE/CLINICAL/OncDRS/')
PROFILE_DATA_PATH = DATA_PATH / 'PROFILE_DATA/'
PROFILE_NOTES_PATH = PROFILE_DATA_PATH / 'CLINICAL_NOTES/'
os.makedirs(PROFILE_NOTES_PATH, exist_ok=True)

# Each worker reads, merges and joins one note file at a time. Workers run a
# 1-thread polars so that N workers x a full-width thread pool each does not
# oversubscribe the node; see note_workers.note_pool_context for why that cap
# has to be set before the pool is built rather than in an initializer.
MAX_WORKERS = min(16, os.cpu_count() or 4)
WORKER_THREADS = 1
MP_CONTEXT = note_pool_context(WORKER_THREADS)

# zstd 15 costs roughly 3.6x the write time of level 10 for ~19% smaller files,
# applied to the largest frames in the pipeline. Exposed here so the tradeoff
# can be measured on real data without editing three call sites. The metadata
# files are written at 10.
NOTES_COMPRESSION_LEVEL = 15

def file_size(path: str | Path) -> int:
    return Path(path).stat().st_size

def human_size(size: int) -> str:
    size = float(size)

    for unit in ("B", "KB", "MB", "GB", "TB"):
        if size < 1024 or unit == "TB":
            return f"{size:.1f} {unit}"
        size /= 1024

text_pulls = {
    'CLINICAL_TEXTS_2024_03' : {'sub_dirs' : None}, 
    'CLINICAL_TEXTS_2025_03' : {'sub_dirs' : None},
    'CLINICAL_TEXTS_2025_11' : {'sub_dirs' : None}, 
    'CLINICAL_TEXTS_2026_03' : {
        'sub_dirs' : ['Discharge Summary Notes', 'Pathology_Cytology Notes', 
                      'Progress Notes', 'Imaging Notes']}
}

image_path_schema = {'RPT_ID' : pl.Int64,
                     'DFCI_MRN' : pl.Int64, 
                     'EVENT_DATE' : pl.String, 
                     'PROC_DESC' : pl.String, 
                     'RPT_TYPE' : pl.String,
                     'RPT_TEXT' : pl.String,
                     'NARRATIVE_TEXT' : pl.String}

prog_disc_schema = {'RPT_ID' : pl.Int64,
                    'DFCI_MRN' : pl.Int64, 
                    'EVENT_DATE' : pl.String,
                    'INP_RPT_TYPE' : pl.String,
                    'PROVIDER_TYPE' : pl.String, 
                    'ENCOUNTER_TYPE_DESC' : pl.String,
                    'RPT_TEXT' : pl.String}

path_meta = pl.read_parquet(PROFILE_NOTES_PATH / 'PATHOLOGY_NOTES_METADATA.parquet')
image_meta = pl.read_parquet(PROFILE_NOTES_PATH / 'IMAGING_NOTES_METADATA.parquet')
prog_meta = pl.read_parquet(PROFILE_NOTES_PATH / 'PROGRESS_NOTES_METADATA.parquet')

path_join_cols = ['RPT_ID', 'DFCI_MRN', 'EVENT_DATE', 'PROC_DESC', 'RPT_TYPE']
image_join_cols = ['RPT_ID', 'DFCI_MRN', 'EVENT_DATE', 'PROC_DESC', 'RPT_TYPE']
prog_join_cols = ['RPT_ID', 'DFCI_MRN', 'EVENT_DATE', 'INP_RPT_TYPE', 'PROVIDER_TYPE', 'ENCOUNTER_TYPE_DESC']


def compile_notes(meta, schema, join_cols, output_name, clean):
    """Re-read every note file for one note type and join its text to `meta`.

    Files are processed largest-first so the long tail lands early and the pool
    drains evenly. executor.map yields in submission order, so the assembled
    frame matches what the serial loop produced.

    Each worker gets only its own metadata slice. Partitioning once up front
    replaces a full scan of `meta` per file -- previously done twice per file,
    since the row-count assert re-filtered to fetch the expected count.
    """
    file_counts = meta['FILE'].value_counts().sort('count', descending=True)
    files_to_extract = file_counts['FILE'].to_list()
    expected_counts = dict(zip(file_counts['FILE'].to_list(), file_counts['count'].to_list()))

    meta_by_file = {key[0]: slice_ for key, slice_ in meta.partition_by('FILE', as_dict=True).items()}
    work_items = [(file_name, meta_by_file[file_name]) for file_name in files_to_extract]

    worker = partial(
        process_note_file,
        schema=schema,
        join_cols=join_cols,
        oncdrs_path=OncDRS_PATH,
        clean=clean,
    )

    df_list = []
    with ProcessPoolExecutor(
        max_workers=MAX_WORKERS,
        mp_context=MP_CONTEXT,
    ) as executor:
        results = executor.map(worker, work_items, chunksize=1)
        for file_name, text_file in tqdm(results, total=len(work_items)):
            assert(len(text_file) == expected_counts[file_name])
            df_list.append(text_file)

    complete_df = pl.concat(df_list, how='vertical')
    assert(len(complete_df) == file_counts['count'].sum())
    complete_df.write_parquet(PROFILE_NOTES_PATH / output_name,
                              compression='zstd', compression_level=NOTES_COMPRESSION_LEVEL)

    source_file_size = sum([file_size(OncDRS_PATH / file_name) for file_name in files_to_extract])
    end_file_size = file_size(PROFILE_NOTES_PATH / output_name)

    print(f'source data size = {human_size(source_file_size)}')
    print(f'end file size = {human_size(end_file_size)}')


start = time()
print(f'starting path files')
compile_notes(path_meta, image_path_schema, path_join_cols, 'PATHOLOGY_NOTES.parquet', clean=True)
print(f'path notes compilation complete in {(time() - start) / 60 : 0.2f} minutes')

start = time()
print(f'starting image files')
compile_notes(image_meta, image_path_schema, image_join_cols, 'IMAGING_NOTES.parquet', clean=True)
print(f'image notes compilation complete in {(time() - start) / 60 : 0.2f} minutes')

start = time()
print(f'starting progress/discharge files')
compile_notes(prog_meta, prog_disc_schema, prog_join_cols, 'PROGRESS_NOTES.parquet', clean=False)
print(f'progress/discharge notes compilation complete in {(time() - start) / 60 : 0.2f} minutes')